# ДЗ: Information Retrieval (BEIR + BM25 + Dense + Hybrid)

Датасет: **FiQA** (BEIR). Реализация на основе семинарских ноутбуков по IR и Qdrant.

## 1. Установка зависимостей

In [ ]:
!pip -q install qdrant-client beir sentence-transformers rank-bm25 pandas

In [ ]:
import os
import re
import time
import random
import warnings

import numpy as np
import pandas as pd
import torch
from collections import defaultdict

from sentence_transformers import SentenceTransformer
from beir import util
from beir.datasets.data_loader import GenericDataLoader
from beir.retrieval.evaluation import EvaluateRetrieval
from rank_bm25 import BM25Okapi
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

warnings.filterwarnings("ignore")

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
device

## 2. Загрузка датасета BEIR FiQA (1 балл)

In [ ]:
data_dir = "./data"
os.makedirs(data_dir, exist_ok=True)

dataset = "fiqa"
url = f"https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{dataset}.zip"

fiqa_path = util.download_and_unzip(url, data_dir)
corpus, queries, qrels = GenericDataLoader(fiqa_path).load(split="test")

print(f"Docs: {len(corpus)}, Queries: {len(queries)}, qrels: {len(qrels)}")

sample_id = next(iter(corpus.keys()))
print({
    "doc_id": sample_id,
    "title": corpus[sample_id].get("title"),
    "text": corpus[sample_id].get("text", "")[:200],
})

## 3. Подготовка датасета (2 балла)

Нормализация → токенизация → чанкинг.

### 3a. Нормализация текста

In [ ]:
def norm_text(s):
    s = s.lower()
    s = s.replace("\xa0", " ").replace("ё", "е")
    s = re.sub(r"\s+", " ", s)
    return s.strip()


def preprocess_text(s):
    s = norm_text(s)
    s = re.sub(r"[^\w\s]", " ", s)
    s = re.sub(r"\s+", " ", s)
    return s.strip()

### 3b. Токенизация

In [ ]:
def tokenize(text):
    return preprocess_text(text).split()


example = corpus[sample_id]
raw = (example.get("title", "") + " " + example.get("text", "")).strip()
print("raw:", raw[:120], "...")
print("tokens:", tokenize(raw)[:15])

### 3c. Чанкинг

In [ ]:
def chunk_text(text, max_words=120, overlap=30):
    words = text.split()
    if len(words) <= max_words:
        return [text]

    chunks = []
    start = 0
    while start < len(words):
        end = min(start + max_words, len(words))
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        if end == len(words):
            break
        start = max(0, end - overlap)
    return chunks

In [ ]:
chunk_corpus = {}
doc_to_chunks = defaultdict(list)

for doc_id, doc in corpus.items():
    text = preprocess_text((doc.get("title", "") + " " + doc.get("text", "")).strip())
    chunks = chunk_text(text, max_words=120, overlap=30)
    for idx, ch in enumerate(chunks):
        chunk_id = f"{doc_id}#c{idx}"
        chunk_corpus[chunk_id] = {
            "text": ch,
            "doc_id": doc_id,
            "title": doc.get("title", ""),
            "chunk_idx": idx,
        }
        doc_to_chunks[doc_id].append(chunk_id)

avg_chunks = np.mean([len(v) for v in doc_to_chunks.values()])
print(f"Total chunks: {len(chunk_corpus)}, avg chunks/doc: {avg_chunks:.2f}")

## 4. Функция оценки качества поиска (1 балл)

In [ ]:
K_VALUES = [1, 3, 5, 10]


def evaluate_beir(qrels, queries_dict, scorer_fn, topk=100, max_queries=None):
    results = {}
    qids = list(queries_dict.keys())
    if max_queries:
        qids = qids[:max_queries]

    start = time.perf_counter()

    for qid in qids:
        q = queries_dict[qid]
        dscores, _ = scorer_fn(q)
        top_items = dict(sorted(dscores.items(), key=lambda x: x[1], reverse=True)[:max(1000, topk)])
        results[qid] = top_items

    duration = time.perf_counter() - start

    evaluator = EvaluateRetrieval()
    ndcg, _map, recall, precision = evaluator.evaluate(qrels, results, K_VALUES)
    return {
        "ndcg": ndcg,
        "map": _map,
        "recall": recall,
        "precision": precision,
        "time_sec": duration,
        "nq": len(qids),
    }


def print_metrics(name, metrics):
    print(f"\n=== {name} ===")
    print(f"Queries: {metrics['nq']}, time: {metrics['time_sec']:.1f}s")
    for k in K_VALUES:
        print(
            f"k={k}: "
            f"NDCG={metrics['ndcg'][f'NDCG@{k}']:.4f}, "
            f"MAP={metrics['map'][f'MAP@{k}']:.4f}, "
            f"Recall={metrics['recall'][f'Recall@{k}']:.4f}, "
            f"P={metrics['precision'][f'P@{k}']:.4f}"
        )

## 5. Sparse поиск: BM25 (2 балла)

In [ ]:
doc_ids = list(corpus.keys())
doc_texts = [
    preprocess_text((corpus[d].get("title", "") + " " + corpus[d].get("text", "")).strip())
    for d in doc_ids
]
tokenized_corpus = [tokenize(t) for t in doc_texts]
bm25 = BM25Okapi(tokenized_corpus)

In [ ]:
def bm25_doc_scores(query, top_k=100):
    scores = bm25.get_scores(tokenize(query))
    top_idx = np.argsort(scores)[::-1][:top_k]
    return {doc_ids[i]: float(scores[i]) for i in top_idx}


def bm25_scorer(query):
    return bm25_doc_scores(query, top_k=100), None

In [ ]:
metrics_bm25 = evaluate_beir(qrels, queries, bm25_scorer, topk=10)
print_metrics("BM25", metrics_bm25)

## 6. Dense поиск: E5 + Qdrant (2 балла)

In [ ]:
EMB_NAME = "intfloat/e5-small-v2"
e5 = SentenceTransformer(EMB_NAME, device=device)

In [ ]:
def e5_encode_texts(texts, is_query=False, batch_size=256):
    prefix = "query: " if is_query else "passage: "
    prepared = [prefix + t.strip() for t in texts]
    vecs = e5.encode(
        prepared,
        batch_size=batch_size,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=True,
    )
    return vecs

In [ ]:
chunk_ids = list(chunk_corpus.keys())
chunk_texts = [chunk_corpus[cid]["text"] for cid in chunk_ids]
chunk_embeddings = e5_encode_texts(chunk_texts, is_query=False, batch_size=256)
dim = chunk_embeddings.shape[1]
print(f"E5 dense dimension: {dim}")

In [ ]:
collection = "fiqa_e5_chunks"
client = QdrantClient(location=":memory:")

client.recreate_collection(
    collection_name=collection,
    vectors_config=VectorParams(size=dim, distance=Distance.COSINE),
)

In [ ]:
BATCH = 512
points = []

for i, cid in enumerate(chunk_ids):
    vec = chunk_embeddings[i].astype(np.float32)
    payload = {
        "chunk_id": cid,
        "doc_id": chunk_corpus[cid]["doc_id"],
        "title": chunk_corpus[cid]["title"],
        "chunk_idx": chunk_corpus[cid]["chunk_idx"],
    }
    points.append(PointStruct(id=i, vector=vec.tolist(), payload=payload))

    if len(points) == BATCH or i == len(chunk_ids) - 1:
        client.upsert(collection_name=collection, points=points)
        points = []

print(f"Uploaded {len(chunk_ids)} chunks to Qdrant")

In [ ]:
def search_dense_qdrant(query, top_k=100):
    qv = e5_encode_texts([query], is_query=True, batch_size=1)[0].astype(np.float32)

    res = client.query_points(
        collection_name=collection,
        query=qv.tolist(),
        limit=top_k,
        with_payload=True,
    )

    doc_scores = defaultdict(float)
    for r in res.points:
        did = r.payload["doc_id"]
        doc_scores[did] = max(doc_scores[did], r.score)
    return doc_scores, res


def dense_scorer(query):
    return search_dense_qdrant(query, top_k=100)

In [ ]:
metrics_dense = evaluate_beir(qrels, queries, dense_scorer, topk=10)
print_metrics("Dense (E5 + Qdrant)", metrics_dense)

## 7. Hybrid поиск: BM25 + Dense (2 балла)

In [ ]:
def hybrid_scores(query, alpha=0.5, top_k=100):
    dscores, _ = search_dense_qdrant(query, top_k=top_k)
    bscores = bm25_doc_scores(query, top_k=top_k)
    all_docs = set(dscores.keys()) | set(bscores.keys())

    def z_norm(d):
        vals = np.array(list(d.values()))
        mu, std = vals.mean(), (vals.std() + 1e-6)
        return {k: (v - mu) / std for k, v in d.items()}

    dz = z_norm(dscores)
    bz = z_norm(bscores)
    fused = {}

    for d in all_docs:
        fused[d] = alpha * dz.get(d, -5.0) + (1 - alpha) * bz.get(d, -5.0)

    fused_top = dict(sorted(fused.items(), key=lambda x: x[1], reverse=True)[:max(1000, top_k)])
    return fused_top


def hybrid_scorer(query):
    return hybrid_scores(query, alpha=0.5, top_k=100), None

In [ ]:
metrics_hybrid = evaluate_beir(qrels, queries, hybrid_scorer, topk=10)
print_metrics("Hybrid (alpha=0.5)", metrics_hybrid)

## 8. Сводное сравнение методов

In [ ]:
def metrics_to_row(method, metrics):
    row = {"method": method, "time_sec": round(metrics["time_sec"], 1)}
    for k in K_VALUES:
        row[f"NDCG@{k}"] = round(metrics["ndcg"][f"NDCG@{k}"], 4)
        row[f"MAP@{k}"] = round(metrics["map"][f"MAP@{k}"], 4)
        row[f"Recall@{k}"] = round(metrics["recall"][f"Recall@{k}"], 4)
        row[f"P@{k}"] = round(metrics["precision"][f"P@{k}"], 4)
    return row


summary_df = pd.DataFrame([
    metrics_to_row("BM25", metrics_bm25),
    metrics_to_row("Dense", metrics_dense),
    metrics_to_row("Hybrid", metrics_hybrid),
])

summary_df

In [ ]:
compare_cols = ["method"] + [f"NDCG@{k}" for k in K_VALUES] + [f"P@{k}" for k in K_VALUES]
summary_df[compare_cols]